This notebook requires pyfabm.
For instructions on how to build and install pyfabm, see https://fabm.net/python

In [ ]:
# Note: "%matplotlib widget" below enables interactive plots but requires https://matplotlib.org/ipympl/
# Alternatively you could use "%matplotlib notebook" (interactive but deprecated) or "%matplotlib inline" (static plots)
%matplotlib widget

import logging
from pathlib import Path

import ipywidgets
import yaml
import pandas as pd

import pyfabm

pyfabm.logger = logging.getLogger('pyfabm')
logging.getLogger('pyfabm').setLevel(logging.WARNING)

In [ ]:
# Show available test cases
# Not all have conserved quantities! Try fabm-examples-npzd-components
testcases = [(path.stem, path) for path in Path("..").glob("*.yaml")]
testcase_dropdown = ipywidgets.Dropdown(
    options=testcases,
    description="Test case:",
    value=Path("../fabm-examples-npzd-components.yaml"),
)
display(testcase_dropdown)

In [ ]:
# Load YAML file and inject check_conservation: true
yaml_file = testcase_dropdown.value
with open(yaml_file) as f:
    cfg = yaml.safe_load(f)
cfg["check_conservation"] = True
model = pyfabm.Model(cfg)

In [ ]:
# Set up the environment under which to check conservation.
# This loads the default testing environment included with FABM.
# You can override specific values by modifying the env dictionary.
with open("../../util/developers/environment.yaml") as f:
    env = yaml.safe_load(f)

model.cell_thickness = env["cell_thickness"]
for dep in model.dependencies:
    dep.value = env.get(dep.name, 0.0)

In [ ]:
# Enumerate conserved quantities, identify instances with associated diagnostics,
# and mark those diagnostics for saving
instances = set()
for conserved_quantity in model.conserved_quantities:
    change_int = f"change_in_{conserved_quantity.name}"
    change_surf = f"change_in_{conserved_quantity.name}_at_surface"
    change_bot = f"change_in_{conserved_quantity.name}_at_bottom"

    # Collect instance prefixes from all three diagnostic pools
    for variables, change_names in [
        (model.interior_diagnostic_variables, [change_int]),
        (model.horizontal_diagnostic_variables, [change_surf, change_bot]),
    ]:
        for v in variables:
            for change_name in change_names:
                if v.name.endswith("/" + change_name):
                    v.save = True
                    instances.add(v.name.rsplit("/", 1)[0])

# Initialize model.
# After this, "save" flags can no longer be modified.
model.start(verbose=False, stop=False)

# Calculate diagnostics
model.get_sources(t=0.0)

totals = model.get_conserved_quantities()
totals[totals == 0.0] = 1.0  # Avoid divide-by-zero errors


def get_value(variables, name, default=0.0):
    try:
        return variables[name].value
    except KeyError:
        return default


# Combine per-instance change in conserved quantities into a single table for display
# Note: rates of change are normalized to the depth-integrated total of the conserved
# quantity, or, in the case of interior change, to that total divided by cell thickness.
rows = []
for conserved_quantity, total in zip(model.conserved_quantities, totals):
    change_int = f"change_in_{conserved_quantity.name}"
    change_surf = f"change_in_{conserved_quantity.name}_at_surface"
    change_bot = f"change_in_{conserved_quantity.name}_at_bottom"

    for instance in sorted(instances):
        fi = get_value(model.interior_diagnostic_variables, f"{instance}/{change_int}")
        fs = get_value(
            model.horizontal_diagnostic_variables, f"{instance}/{change_surf}"
        )
        fb = get_value(
            model.horizontal_diagnostic_variables, f"{instance}/{change_bot}"
        )
        rows.append(
            {
                "instance": instance or "(root)",
                "conserved quantity": conserved_quantity.long_name,
                "units": conserved_quantity.units,
                "interior change": fi / (total / env["cell_thickness"]),
                "surface change": fs / total,
                "bottom change": fb / total,
            }
        )
if not rows:
    print(f"No conserved quantities found for {yaml_file}.")
else:
    df = pd.DataFrame(rows)
    pd.set_option("display.max_rows", None)

    def color_cells(val):
        if abs(val) > 1e-20:
            return "background-color: #FF7276"
        else:
            return "background-color: lightgreen"

    subset = ["interior change", "surface change", "bottom change"]
    styled_df = (
        df.style.map(color_cells, subset)
        .format("{:.3e}", subset=subset)
        .hide(axis="index")
    )
    display(styled_df)